In [0]:
%run "../00_config"

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime, timezone

##Load Bronze weather

In [0]:
df_bronze = spark.table(BRONZE_WEATHER)
display(df_bronze)

##Clean and transform

In [0]:
df_silver = (df_bronze
    # Cast types
    .withColumn("apparent_temperature_max",    F.col("apparent_temperature_max").cast(DoubleType()))
    .withColumn("precipitation_sum",           F.col("precipitation_sum").cast(DoubleType()))
    .withColumn("temperature_max",             F.col("temperature_max").cast(DoubleType()))
    .withColumn("temperature_min",             F.col("temperature_min").cast(DoubleType()))
    .withColumn("wind_direction_dominant",     F.col("wind_direction_dominant").cast(DoubleType()))
    .withColumn("windspeed_max",               F.col("windspeed_max").cast(DoubleType()))
    .withColumn("latitude",                    F.col("latitude").cast(DoubleType()))
    .withColumn("longitude",                   F.col("longitude").cast(DoubleType()))

    # Parse order_date to DateType
    .withColumn("date",
        F.to_date(F.col("date"), "yyyy-MM-dd")
    )

    # Replace empty strings with null
    .withColumn("city",
        F.when(F.col("city") == "", F.lit(None))
        .otherwise(F.col("city"))
    )

     # Standardize order_status
    .withColumn("city",
        F.initcap(F.col("city"))
    )

    # Clean date
    .withColumn("_snapshot_date", (F.to_date(F.col("_snapshot_date"), "dd-MM-yyyy")))

    # Filter out orders with no order_id, quantity and unit_price_sar
    .filter(F.col("city") != "")
    .filter(F.col("city").isNotNull())
    .filter(F.col("date").isNotNull())

    # Add ingestion metadata
    .withColumn("_silver_ingested_at", F.to_timestamp(F.lit(datetime.now(timezone.utc).isoformat())))

    # Deduplicate rows by city and date
    .dropDuplicates(["city", "date"])

    # Final column selection
    .select(
        "city",
        "latitude",
        "longitude",
        "date",
        "temperature_max",
        "temperature_min",
        "apparent_temperature_max",
        "precipitation_sum",
        "windspeed_max",
        "wind_direction_dominant",
        "_snapshot_date",
        "_silver_ingested_at"
    )
)

print(f"Silver rows after cleaning: {df_silver.count()}")
display(df_silver)

##Write to silver

In [0]:
(df_silver
   .write
   .format("delta")
   .mode("overwrite")
   .saveAsTable(SILVER_WEATHER)
)
print(f"✅ {df_silver.count()} weather records written to {SILVER_WEATHER}")

##Validate

In [0]:
df_check = spark.table(SILVER_WEATHER)
print(f"Total rows     : {df_check.count()}")
print(f"Distinct cities: {df_check.select('city').distinct().count()}")
print(f"Date range     : {df_check.agg(F.min('date'), F.max('date')).collect()[0]}")
df_check.printSchema()
df_check.select(
   "city", "date", "temperature_max",
   "temperature_min", "precipitation_sum", "windspeed_max"
).show(5, truncate=30)

In [0]:
%sql
SELECT distinct city FROM saudi_retail_catalog.silver.silver_weather;